In [1]:
import os

## Write ```Dockerfile``` to local drive

In [2]:
%%writefile Dockerfile

FROM python:3.9
    
# update package manager
RUN apt-get update

# update pip
RUN pip install --upgrade pip

# copy requirements
COPY requirements.txt .

# install dependencies
RUN pip install -r requirements.txt

# copy script into container
COPY script.py .

# run script when image is run
CMD ["python3", "script.py"]

Writing Dockerfile


## Write ```requirements.txt```

In [3]:
%%writefile requirements.txt

pyarrow==9.0.0
fsspec==2022.10.0
s3fs==2022.10.0

pandas==1.2.4

Writing requirements.txt


## Write ```script.py``` to local drive

In [4]:
%%writefile script.py

import os
import pandas as pd

# df info
def get_df_info(df):
    int_nrows, int_ncols = df.shape
    print(f'Rows: {int_nrows}; Columns: {int_ncols}')
    print('')

# id no var cols
def find_no_variance_columns(df):
    # sample
    df_tmp = df.sample(
        frac=0.01, 
        random_state=42,
    )
    get_df_info(df_tmp)
    
    ser_nunique = df_tmp.nunique()
    ser_nunique = ser_nunique[ser_nunique==1]
    list_cols = list(ser_nunique.index)
    print(f'There are {len(list_cols)} no variance features in the preliminary stage:')
    print('')
    # full
    ser_nunique = df[list_cols].nunique()
    ser_nunique = ser_nunique[ser_nunique==1]
    list_cols = list(ser_nunique.index)
    print(f'There are {len(list_cols)} columns with no variance:')
    return list_cols

# get redundant feartures
def find_redundant_columns(df):
    # sample
    df_tmp = df.sample(
        frac=0.01, 
        random_state=42,
    )
    get_df_info(df_tmp)

    # transpose the dataframe and check for duplicate rows
    ser_duplicates = df_tmp.T.duplicated() * 1
    ser_duplicates = ser_duplicates[ser_duplicates==1]
    list_cols_a = list(ser_duplicates.index)
    print(f'There are {len(list_cols_a)} redundant features in the preliminary stage:')
    print('')

    list_cols = []
    a = 0
    for a, col_a in enumerate(list_cols_a):
        # add 1 to a
        a += 1
        if col_a not in list_cols:
            # create new list
            list_cols_b = list_cols_a[a:]
            for col_b in list_cols_b:
                if df[col_a].equals(df[col_b]):
                    list_cols.append(col_b)   
        else:
            pass
    print(f'There are {len(list_cols)} redundant columns:')
    # return
    return list_cols

# constants
str_project = '20231010-gen-xii'
str_dirname_output = './output'

str_id = 'uniqueid'
str_target = 'target'
str_datecol = 'applicationdate__app'
str_model = '02_pricing_pd'

# check for no var and redundant features in each data set
list_cols_drop = []
for a, str_df in enumerate(['train','valid','test']):
    # message
    print(f'Importing {str_df} data...')
    # import the data
    str_filename = f'df_{str_df}_noleaks_pre.gzip'
    str_uri = f's3://{str_project}/{str_model}/02_model/00_preprocessing/02_make_dfs/{str_filename}'
    df = pd.read_parquet(str_uri)
    get_df_info(df)
    # logic
    if a == 0:
        list_cols_start = list(df.columns)
        
    # check for no variance
    list_cols = find_no_variance_columns(df)
    # extend
    list_cols_drop.extend(list_cols)
    # save as df
    df_tmp = pd.DataFrame({'feature': list_cols})
    str_filename = f'df_no_var_{str_df}.csv'
    str_uri = f's3://{str_project}/{str_model}/02_model/00_preprocessing/03_get_list_of_cols/{str_filename}'
    df_tmp.to_csv(str_uri, index=False)
    
    # check for redundancy
    list_cols = find_redundant_columns(df)
    # extend
    list_cols_drop.extend(list_cols)
    # save as df
    df_tmp = pd.DataFrame({'feature': list_cols})
    str_filename = f'df_redundant_{str_df}.csv'
    str_uri = f's3://{str_project}/{str_model}/02_model/00_preprocessing/03_get_list_of_cols/{str_filename}'
    df_tmp.to_csv(str_uri, index=False)
    
    # delete
    del df

# rm dups
list_cols_drop = list(dict.fromkeys(list_cols_drop))
print(f'There are {len(list_cols_drop)} columns to drop:')
print(list_cols_drop)

# save to s3
# rm feats
list_cols_final = [col for col in list_cols_start if col not in list_cols_drop]
# make df
df = pd.DataFrame({'feature': list_cols_final})
# write to s3
str_filename = 'df_cols_final.csv'
str_uri = f's3://{str_project}/{str_model}/02_model/00_preprocessing/03_get_list_of_cols/{str_filename}'
df.to_csv(str_uri, index=False)

Writing script.py


## Build and push to ECR

In [5]:
%%sh

# image name
image=genxii-pd-listcols

# Get the account number associated with the current IAM credentials
account=$(aws sts get-caller-identity --query Account --output text)

# did we have an error?
if [ $? -ne 0 ]
then
    exit 255
fi

# Get the region defined in the current configuration (default to us-west-2 if none defined)
region=$(aws configure get region)
region=${region:-us-west-2}

# get destination of repo
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# If the repository doesn't exist in ECR, create it.
aws ecr describe-repositories --repository-names "${image}" > /dev/null 2>&1

# if it doesnt exist...create it
if [ $? -ne 0 ]
then
    aws ecr create-repository --repository-name "${image}" > /dev/null
fi

# Get the login command from ECR and execute it directly
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# Build the docker image locally with the image name and then push it to ECR
# with the full name.

# build and add tag
docker build  -t ${image} .
docker tag ${image} ${fullname}
# push to ecr
docker push ${fullname}

WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded
Sending build context to Docker daemon   21.5kB
Step 1/7 : FROM python:3.9
 ---> 7ef94ac333fa
Step 2/7 : RUN apt-get update
 ---> Using cache
 ---> 529128453704
Step 3/7 : RUN pip install --upgrade pip
 ---> Using cache
 ---> 8f56313abc92
Step 4/7 : COPY requirements.txt .
 ---> 6508c8046f55
Step 5/7 : RUN pip install -r requirements.txt
 ---> Running in 4af1720ef3c7
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.8/138.8 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 90.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 80.5 

Removing intermediate container 4af1720ef3c7
 ---> a77ba1ad6586
Step 6/7 : COPY script.py .
 ---> 90d1d8f9551e
Step 7/7 : CMD ["python3", "script.py"]
 ---> Running in 731f84e802bc
Removing intermediate container 731f84e802bc
 ---> 481493a1048f
Successfully built 481493a1048f
Successfully tagged genxii-pd-listcols:latest
The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-pd-listcols]
221389aee578: Preparing
94b47eff471d: Preparing
b60fe4b89d50: Preparing
7f7a9ee63288: Preparing
781f058a9424: Preparing
78ecb2a2f011: Preparing
84062ebc4cf5: Preparing
2180aea5f54b: Preparing
86388e04a96b: Preparing
893507f6057f: Preparing
2353f7120e0e: Preparing
51a9318e6edf: Preparing
c5bb35826823: Preparing
94b47eff471d: Waiting
b60fe4b89d50: Waiting
7f7a9ee63288: Waiting
781f058a9424: Waiting
78ecb2a2f011: Waiting
84062ebc4cf5: Waiting
2180aea5f54b: Waiting
86388e04a96b: Waiting
893507f6057f: Waiting
2353f7120e0e: Waiting
51a9318e6edf: Waiting
c5bb35826823: Waiting
22138

## Clean-up

In [6]:
# rm files
for str_file in ['Dockerfile','requirements.txt','script.py']:
    try:
        os.remove(f'./{str_file}')
    except:
        pass